# Imports

In [ ]:
import os
import re
import tifffile
import skimage
import scipy
import cv2 as cv
import skan
import pandas as pd
import numpy as np
from scipy.ndimage import distance_transform_edt

/Users/ar2067/opt/anaconda3/envs/python_analysis/lib/python3.12/site-packages/pydantic/_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


# Functions

In [ ]:

def first_nonzero_pixel_left(imdata):
    for i in range(imdata.shape[1]):
        if not np.any(imdata[:,i]):
            continue
        return (imdata[:,i].argmax(),i)

def first_nonzero_pixel_right(imdata):
    for i in range(imdata.shape[1]-1,-1,-1):
        if not np.any(imdata[:,i]):
            continue
        return (imdata[:,i].argmax(),i)
        

def make_slice_mask(imdata,start,thickness):
    """
    This takes an image consisting of a line representing the top of the cells 
    and returns a mask defining a slice of the tissue that starts 'start' 
    pixels below the line and is 'thickness' pixels thick.
    
    imadata : 2D numpy array
        The numpy array containing a one-pixel-thick line representing the top 
        of the cells.
    start : int
        The number of pixels below the top from which you want your mask to 
        start.
    thickness : int
        The total thickness of the mask.
    """
    
    LY,LX = first_nonzero_pixel_left(imdata)
    RY,RX = first_nonzero_pixel_right(imdata)

    imdata2 = imdata.copy()
    imdata2[LY,:LX] = 1
    imdata2[RY,RX:] = 1

    imdata2 = imdata2.astype('uint8')
    
    start_kernel = np.ones((start,start),np.uint8)
    start_mask = cv.dilate(imdata2,start_kernel)

    thickness_kernel = np.ones((thickness+start,thickness+start),np.uint8)
    thickness_mask = cv.dilate(imdata2,thickness_kernel)   

    out_mask = thickness_mask - start_mask

    for i in range(out_mask.shape[1]): 
        SY = imdata2[:,i].argmax()
        out_mask[:SY,i] = 0

    out_mask[:,:LX] = 0
    out_mask[:,RX:] = 0

    return out_mask


def parse_monolayer_crosssection_segmentation(seg,return_J_cell_neighbours_dict=False):
    """
    You give this a boolean numpy array which is a one pixel thick 
    segmentation marking the junctions of a cross-section image of a monolayer 
    of cells. It parses the segmentation to return lots of useful versions of 
    the segmentation (see returns below).

    Returns
    ---------
    seg_JIDs : numpy array
        The segmentation but pixel values are the ID of the junction. This 
        starts at 1 rather than 0 since 0 is the background.
    seg_count : numpy array
        The segmentation but pixel values are the pixel index along that 
        junction. This starts at 1 rather than 0 since 0 is the background. 
        Counting starts at the end with the lowest y-value. Pixel index can 
        be used as proxy for distance along junction if you don't care about 
        diagonal vs straight.
    label_cells : numpy array
        The inverse segmentation (junctions have pixel value zero) but cell 
        pixels values are the cell IDs.
    seg_j_class : numpy array
        The segmentation but pixel values are the class of the junction.
        1 : top border junction
        2 : bottom border junction
        3 : interior junction
        4 : a border junction on a cell that has just 2 junctions, i.e. maybe 
            end cell, should probably be ignored.
    cells_class : numpy array
        Like label_cells but values at cells are 2 for interior cell or 3 for 
        cells that have only 2 junctions. And 0 is junctions and 1 is 
        background.
    """
    # get rid of edge borders because it complicates things
    seg[:,0] = False
    seg[:,-1] = False
    seg[0,:] = False
    seg[-1,:] = False    

    skel_object = skan.Skeleton(seg)

    # keep pruning the skeleton until no dangling edges
    while np.any(skel_object.degrees==1):
        degree_one_pixels = np.where(skel_object.degrees==1)[0]
        prune_JIDs = [np.where([pix_ID in path for path in skel_object.paths_list()])[0][0] for pix_ID in degree_one_pixels]
        skel_object = skel_object.prune_paths(prune_JIDs)

    seg = skel_object.skeleton_image.copy()
    paths_list = skel_object.paths_list()

    seg_JIDs = np.zeros(seg.shape,dtype='uint16')
    for i,path in enumerate(paths_list):
        for pixelID in path:
            y,x = skel_object.coordinates[pixelID]
            seg_JIDs[y,x] = i+1  

    seg_count = np.zeros(seg.shape,dtype='uint16')
    for path in paths_list:
        for i,pixelID in enumerate(path):
            y,x = skel_object.coordinates[pixelID]
            seg_count[y,x] = i+1      

    label_cells = skimage.measure.label(~seg,connectivity=1)
    regions = skimage.measure.regionprops(label_cells)[1:]

    # the key is the cell ID and the value is the (y,x) of the centroid
    cell_centres_dict = {}
    for r in regions:
        cell_centres_dict[r.label] = r.centroid    
    
    background_ID = np.unique(label_cells)[np.argmax([np.sum(label_cells==lab) for lab in np.unique(label_cells)])]
    
    cell_IDs = np.unique(label_cells)
    cell_IDs = cell_IDs[cell_IDs!=0]
    cell_IDs = cell_IDs[cell_IDs!=background_ID]

    node_pixel_IDs = np.where(skel_object.degrees==3)[0]

    # an image like seg_JIDs but for node pixels, i.e. the pixel value is zero except as nodes where it is the ID of the node
    seg_nodeIDs = np.zeros_like(seg_count)
    for nodeID in node_pixel_IDs:
        y,x = skel_object.coordinates[nodeID]
        seg_nodeIDs[y,x] = nodeID    
        
    # the keys are junction IDs and the values are a list of the 2 cell IDs that this junction lies between
    J_cell_neighbours_dict = {}
    for J_ID in np.unique(seg_JIDs)[1:]:
        im = np.zeros_like(seg)
        im[seg_JIDs==J_ID] = True
        im[seg_nodeIDs!=0] = 0
        kernel = np.ones((3,3),np.uint8)
        im_dil = cv.dilate(im.astype('uint8'),kernel)  
        cell_IDs2 = label_cells[im_dil.astype('bool')]
        cell_IDs2 = cell_IDs2[cell_IDs2!=0]
        value1 = scipy.stats.mode(cell_IDs2,keepdims=True).mode[0]
        cell_IDs2 = cell_IDs2[cell_IDs2!=value1]
        value2 = scipy.stats.mode(cell_IDs2,keepdims=True).mode[0]
        J_cell_neighbours_dict[J_ID] = [value1,value2]

    # the keys are cell IDs and the values are a list of the junction IDs that belong to that cell
    cell_J_neighbours_dict = {}
    for C_ID in cell_IDs:
        for J_ID,C_IDs in J_cell_neighbours_dict.items():
            if C_ID in C_IDs:
                if C_ID in cell_J_neighbours_dict.keys():
                    cell_J_neighbours_dict[C_ID].append(J_ID)
                else:
                    cell_J_neighbours_dict[C_ID] = [J_ID]   

    # the keys are node pixel IDs and the values are a list of the 3 junction IDs that meet at this node
    node_J_neibours_dict = {}
    for node_ID in node_pixel_IDs:
        for i,path in enumerate(paths_list):
            if node_ID in path:
                if node_ID in node_J_neibours_dict:
                    node_J_neibours_dict[node_ID].append(i+1)
                else:
                    node_J_neibours_dict[node_ID] = [i+1]
    
    # the keys are junction IDs and the values are a list of the 2 node IDs that are at either end of the junction
    J_nodes_dict = {}
    for J_ID in np.unique(seg_JIDs)[1:]:
        for node_ID,node_J_IDs in node_J_neibours_dict.items():
            if J_ID in node_J_IDs:
                if J_ID in J_nodes_dict.keys():
                    J_nodes_dict[J_ID].append(node_ID)
                else:
                    J_nodes_dict[J_ID] = [node_ID]
    
    # the keys are junction IDs and the values are a list of the 4 junction IDs that are neighbours
    J_J_neibours_dict = {}
    for J_ID in np.unique(seg_JIDs)[1:]:
        for node_ID in J_nodes_dict[J_ID]:
            Js1 = node_J_neibours_dict[node_ID].copy()
            Js1.remove(J_ID)
            if J_ID in J_J_neibours_dict:
                J_J_neibours_dict[J_ID] += Js1
            else:
                J_J_neibours_dict[J_ID] = Js1
            

    # the keys are junction IDs and the values whether this junction is a border junction (i.e. if False then it lies between 2 cells)
    border_J_Q_dict = {}
    for J_ID in np.unique(seg_JIDs)[1:]:
        if background_ID in J_cell_neighbours_dict[J_ID]:
            border_J_Q_dict[J_ID] = True
        else:
            border_J_Q_dict[J_ID] = False
    
    # an image of just the border junctions. The pixel values are the junction IDs
    seg_border_Js = np.zeros_like(seg_JIDs)
    for J_ID in np.unique(seg_JIDs)[1:]:
        if border_J_Q_dict[J_ID]:
            for pixelID in paths_list[J_ID-1]:
                y,x = skel_object.coordinates[pixelID]
                seg_border_Js[y,x] = J_ID        

    # an image of just the cell-cell junctions. The pixel values are the junction IDs
    seg_cell_cell_Js = np.zeros_like(seg_JIDs)
    for J_ID in np.unique(seg_JIDs)[1:]:
        if not border_J_Q_dict[J_ID]:
            for pixelID in paths_list[J_ID-1]:
                y,x = skel_object.coordinates[pixelID]
                seg_cell_cell_Js[y,x] = J_ID 
    
    border_sequence = []
    
    border0 = np.unique(seg_border_Js)[1:][0]
    border_sequence.append(border0)
    border1 = border0
    
    border_neighbours = [J for J in J_J_neibours_dict[border1] if border_J_Q_dict[J]]
    border_neighbours = [J for J in border_neighbours if J!=border1]
    border2 = border_neighbours[0]
    
    while border2!=border0:
        border_sequence.append(border2)
        border_neighbours = [J for J in J_J_neibours_dict[border2] if border_J_Q_dict[J]]
        border_neighbours = [J for J in border_neighbours if J!=border1]
        border1 = border2
        border2 = border_neighbours[0]
    
    upQ = [True if i<len(border_sequence)//2 else False for i in range(len(border_sequence))]
    
    full_up_measures = []
    for i in range(len(border_sequence)):
        up_measures = []
        for J_ID,up in zip(border_sequence,np.roll(upQ,i)):
            if up:
                y1,x1 = skel_object.coordinates[J_nodes_dict[J_ID][0]]
                y2,x2 = skel_object.coordinates[J_nodes_dict[J_ID][1]]
                if x2<x1:
                    y2,x2 = skel_object.coordinates[J_nodes_dict[J_ID][0]]
                    y1,x1 = skel_object.coordinates[J_nodes_dict[J_ID][1]] 
                    
                cell1 = [c for c in J_cell_neighbours_dict[J_ID] if c!=background_ID][0]
                yc,xc = cell_centres_dict[cell1]            
        
                measure = ((y2+y1)/2) - yc
                up_measures.append(measure)
        full_up_measures.append(sum(up_measures))
                
    min_i = np.argmin(full_up_measures)
    seg_top_Js = np.zeros_like(seg_JIDs)
    seg_bottom_Js = np.zeros_like(seg_JIDs)
    for J_ID,up in zip(border_sequence,np.roll(upQ,min_i)):
        if up:
            seg_top_Js[seg_JIDs==J_ID] = J_ID
        else:
            seg_bottom_Js[seg_JIDs==J_ID] = J_ID
    
    # the pixel value along each junction is its class:
    # 1 : top border junction
    # 2 : bottom border junction
    # 3 : interior junction
    # 4 : a border junction on a cell that has just 2 junctions, i.e. maybe end cell, should probably be ignored
    seg_j_class = np.zeros_like(seg_JIDs)
    for J_ID in np.unique(seg_JIDs)[1:]:
        cellsJ = [C for C in J_cell_neighbours_dict[J_ID] if C!=background_ID]
        if len(cellsJ)==1 and len(cell_J_neighbours_dict[cellsJ[0]])==2:
            seg_j_class[seg_JIDs==J_ID] = 4    
        elif J_ID in seg_top_Js:
            seg_j_class[seg_JIDs==J_ID] = 1
        elif J_ID in seg_bottom_Js:
            seg_j_class[seg_JIDs==J_ID] = 2
        elif not border_J_Q_dict[J_ID]:
            seg_j_class[seg_JIDs==J_ID] = 3
        else:
            raise Exception('Junction class not found for junction: '+str(J_ID))   

    cells_class = label_cells.copy()
    for lab in np.unique(label_cells)[1:]:
        bad_cell = False
        if lab==background_ID:
            continue
        for J_ID in cell_J_neighbours_dict[lab]:
            if np.all(seg_j_class[seg_JIDs==J_ID]==4):
                bad_cell = True
        if bad_cell:
            cells_class[cells_class==lab] = 3
        else:
            cells_class[cells_class==lab] = 2
                
    out_tuple = (seg_JIDs,seg_count,label_cells,seg_j_class,cells_class)
    if return_J_cell_neighbours_dict:
        out_tuple += (J_cell_neighbours_dict,)
    return out_tuple


def grey_dilate_conserve_brightness(arr,size=5):
    """
    Normal greyscale dilation makes images brighter since it propagates the 
    maximum pixel in the neighbourhood (i.e. in the structure element). This 
    does greyscale dilation without making things brighter. It only works on 
    semi-boolean arrays, i.e. most pixels are zero but the non-zero parts are 
    greyscale. It allows borders to grow with pixel values that are really at 
    the border rather than getting dominated by the brightest ones. It works 
    by taking the average of a greyscale dilation and greyscale erosion in 
    which the eroded image has zeros replaces by the max pixel value of the 
    data type before erosion (then changed back to zero after).
    """

    arr2 = arr.copy()
    arr2[arr2==0] = np.iinfo(arr.dtype).max
    
    arr_dil = scipy.ndimage.grey_dilation(arr, size=(size,size))
    arr2_er = scipy.ndimage.grey_erosion(arr2, size=(size,size))
    
    arr2_er[arr2_er==np.iinfo(arr.dtype).max] = 0
    
    arr_av = np.mean([arr_dil,arr2_er], axis=0)
    
    # Round the values to the nearest integer
    return np.round(arr_av).astype(int) 


def jagged_list_to_array(the_list,pad_with=np.nan):
    """
    This converts a jagged list of sublists to a numpy array by padding at the end of the sublists to make non-jagged.
    """
    
    # Determine the maximum length of sublists
    max_len = max(len(sublist) for sublist in the_list)
    
    # Pad shorter sublists with np.nan
    padded_sublists = [sublist + [np.nan]*(max_len - len(sublist)) for sublist in the_list]
    
    # Convert to numpy array
    return np.array(padded_sublists)


def make_middle_third_mask(junction_class_mask,perc_start=33,perc_end=66):
    """
    Give this a mask of junctions where 1 is top line, 2 is bottom line and 4 
    is bad cells at end and it returns a mask that is just the area between 
    1/3rd and 2/3rds between the top and bottom line.
    """
    dist_tr_top = distance_transform_edt(np.invert(junction_class_mask==1))
    dist_tr_bottom = distance_transform_edt(np.invert(junction_class_mask==2))
    fraction_between_mask = dist_tr_top/(dist_tr_top + dist_tr_bottom)
    
    middle_third_mask = np.logical_and(fraction_between_mask>perc_start/100,fraction_between_mask<=perc_end/100)
    good_cells_filled_mask = scipy.ndimage.binary_fill_holes(np.logical_and(junction_class_mask!=4,junction_class_mask!=0))
    middle_third_mask = np.logical_and(middle_third_mask,good_cells_filled_mask)  
    return middle_third_mask


def masks2rgb(mask1,mask2,mask3=None):
    if not isinstance(mask3,np.ndarray):
        mask3 = np.zeros_like(mask1)
    return np.concatenate((mask1[:,:,np.newaxis],mask2[:,:,np.newaxis],mask3[:,:,np.newaxis]),axis=2).astype('uint8')*255

# Apply analysis to many images

In [4]:
root = r''

root_names = np.unique([re.sub('_(dex|memb)','',fn) for fn in os.listdir(root)])

root_names = [rn for rn in root_names if rn[0]!='.']

filename_pairs = [sorted([fn for fn in os.listdir(root) if rn==re.sub('_(dex|memb)','',fn)]) for rn in root_names]

In [ ]:
j_dil_k_size = 11

im_thresh_perc = 96

region_start_perc = 25
region_end_perc = 75

all_conditions = []
all_measures = []
all_fractions = []
all_im_data_thr = []
all_lateral_third_js_dil = []
all_seg_j_class = []

for i,fnp in enumerate(filename_pairs):
    print(i)
    seg = tifffile.imread(os.path.join(root,fnp[1]))
    seg = skimage.morphology.skeletonize(seg.astype('bool'))
    im_data = tifffile.imread(os.path.join(root,fnp[0]))

    condition = re.search('^(160_CTRL|160_FAKi|160_PI3Ki|3.5)_',fnp[0]).group(1)
    all_conditions.append(condition)
    print(all_conditions)

    seg_JIDs,seg_count,label_cells,seg_j_class,cells_class,J_cell_neighbours_dict = parse_monolayer_crosssection_segmentation(seg,return_J_cell_neighbours_dict=True)

    all_seg_j_class.append(seg_j_class)
    
    good_cells_filled = scipy.ndimage.binary_fill_holes(np.logical_and(seg_j_class!=4,seg_j_class!=0))
    
    middle_third = make_middle_third_mask(seg_j_class,region_start_perc,region_end_perc)
    
    internal_js_dil = scipy.ndimage.grey_dilation(seg_j_class==3, size=(j_dil_k_size,j_dil_k_size))
    lateral_third_js_dil = np.logical_and(internal_js_dil,middle_third)

    all_lateral_third_js_dil.append(lateral_third_js_dil)
    
    all_fractions.append(np.sum(lateral_third_js_dil)/np.sum(middle_third))
    
    im_data_thr = np.zeros_like(im_data)
    thresh1 = np.percentile(im_data[good_cells_filled],im_thresh_perc)
    im_data_thr[im_data>thresh1] = 1
    im_data_thr = im_data_thr.astype('bool')

    all_im_data_thr.append(im_data_thr)
    
    all_measures.append(np.sum(np.logical_and(im_data_thr,lateral_third_js_dil))/np.sum(lateral_third_js_dil))
    



dict = {'condition':all_conditions,'measures':all_measures,'fractions':all_fractions}
df_ibi = pd.DataFrame(dict)
print(df_ibi)